# 🇧🇩 Bengali NLP System — Load from Drive & Launch UI

**No retraining needed.** This notebook:
1. Mounts Google Drive
2. Loads all saved models from `My Drive/NLP_Project_Models/`
3. Launches the full Gradio UI

**Drive folder structure expected:**
```
NLP_Project_Models/
  ├── bilstm.pt
  ├── bilstm_attn.pt
  ├── cnn_bilstm.pt
  ├── embedding.pt
  ├── vocab.pkl
  ├── tfidf_summarizer.pkl
  ├── banglabert_ft/        (HuggingFace model folder)
  ├── xlmr_base/            (HuggingFace model folder)
  ├── zero_shot/            (HuggingFace model folder)
  ├── bert_summarizer/      (HuggingFace model folder)
  ├── bilstm_summarizer/    (HuggingFace model folder)
  └── mt5_summarizer/       (HuggingFace model folder)
```
> Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
# Cell 2 ─ Install packages (~2 min)
import subprocess, sys
pkgs = [
    'transformers>=4.36', 'accelerate>=0.26', 'sentencepiece', 'protobuf',
    'pytesseract', 'Pillow',
    'pdfplumber', 'python-docx',
    'gradio>=4.0',
    'scikit-learn', 'torch', 'tqdm',
]
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade']+pkgs, check=False)
subprocess.run(['apt-get','install','-y','-q',
    'tesseract-ocr','tesseract-ocr-ben','tesseract-ocr-eng',
    'poppler-utils'], check=False, capture_output=True)
print('✅ Packages installed')


✅ Packages installed


In [ ]:
# Cell 3 ─ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/NLP_Project_Models'
assert os.path.exists(DRIVE_DIR), f'❌ Folder not found: {DRIVE_DIR}'
print(f'✅ Drive mounted. Contents of {DRIVE_DIR}:')
for f in sorted(os.listdir(DRIVE_DIR)):
    print(f'   {f}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Contents of /content/drive/MyDrive/NLP_Project_Models:
   banglabert_ft
   bert_summarizer
   best_model.pth
   bilstm.pt
   bilstm_attn.pt
   bilstm_summarizer
   cnn_bilstm.pt
   embedding.pt
   mt5_summarizer
   summarizer
   tfidf_summarizer.pkl
   vocab.pkl
   xlmr_base
   zero_shot


In [ ]:
!pip install rouge-score

In [ ]:
import re
import numpy as np
import pickle
import torch
from rouge_score import rouge_scorer as rs

# Example usage
_scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

def rouge_eval(pred, ref):
    """Compute ROUGE scores for single pair of strings"""
    return _scorer.score(ref, pred)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


def clean_bengali_text(text):
    """Remove non-Bengali characters and normalize whitespace."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\u0980-\u09FF\s।,!?]', '', text)
    return text.strip()


def sentence_tokenize_bengali(text):
    """Split Bengali text into sentences on Bengali/English full stops."""
    sentences = re.split(r'[।.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences


# ── ROUGE helper (optional – used in batch evaluation) ──────────────────────
from rouge_score import rouge_scorer as rs

_scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

def evaluate_rouge(predictions, references):
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not ref:
            continue
        s = _scorer.score(ref, pred)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {
        'ROUGE-1': round(np.mean(r1), 4),
        'ROUGE-2': round(np.mean(r2), 4),
        'ROUGE-L': round(np.mean(rl), 4),
    }


print('Utilities ready.')

Device: cuda
Utilities ready.


In [ ]:
MODELS_DIR = '/content/drive/MyDrive/NLP_Project_Models/summarizer'

In [ ]:
# ── Class definition must be present so pickle can deserialise the object ───
from sklearn.feature_extraction.text import TfidfVectorizer

class TFIDFSummarizer:
    def __init__(self, top_n=3):
        self.top_n = top_n
        self.vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))

    def fit(self, corpus):
        self.vectorizer.fit(corpus)
        return self

    def summarize(self, text):
        sentences = sentence_tokenize_bengali(text)
        if len(sentences) <= self.top_n:
            return ' '.join(sentences)
        mat    = self.vectorizer.transform(sentences)
        scores = np.array(mat.sum(axis=1)).flatten()
        top    = sorted(np.argsort(scores)[-self.top_n:])
        return ' '.join([sentences[i] for i in top])

    @staticmethod
    def load(path):
        with open(path, 'rb') as f:
            return pickle.load(f)


tfidf_model = TFIDFSummarizer.load(f'{MODELS_DIR}/tfidf_summarizer.pkl')
print('TF-IDF model loaded.')

TF-IDF model loaded.


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

class TextRankSummarizer:
    def __init__(self, top_n=3):
        self.top_n = top_n
        self.vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))

    def fit(self, corpus):
        self.vectorizer.fit(corpus)
        return self

    def summarize(self, text):
        sentences = sentence_tokenize_bengali(text)
        if len(sentences) <= self.top_n:
            return ' '.join(sentences)
        mat = self.vectorizer.transform(sentences).toarray()
        sim = cosine_similarity(mat)
        np.fill_diagonal(sim, 0)
        G = nx.from_numpy_array(sim)
        try:
            scores = nx.pagerank(G, max_iter=300)
        except nx.PowerIterationFailedConvergence:
            scores = {i: 1/len(sentences) for i in range(len(sentences))}
        ranked = sorted(scores, key=scores.get, reverse=True)[:self.top_n]
        return ' '.join([sentences[i] for i in sorted(ranked)])

    @staticmethod
    def load(path):
        with open(path, 'rb') as f:
            return pickle.load(f)


textrank_model = TextRankSummarizer.load(f'{MODELS_DIR}/textrank_summarizer.pkl')
print('TextRank model loaded.')

TextRank model loaded.


In [ ]:
import torch.nn as nn

# ── CharVocab helper (needed to encode sentences) ───────────────────────────
class CharVocab:
    def __init__(self, max_vocab=5000):
        self.max_vocab = max_vocab
        self.char2idx  = {'<PAD>': 0, '<UNK>': 1}

    def fit(self, texts):
        from collections import Counter
        ctr = Counter(c for text in texts for c in text)
        for ch, _ in ctr.most_common(self.max_vocab - 2):
            self.char2idx[ch] = len(self.char2idx)
        return self

    def encode(self, text, max_len=100):
        ids = [self.char2idx.get(c, 1) for c in text[:max_len]]
        ids += [0] * (max_len - len(ids))
        return ids


# ── Model architecture ───────────────────────────────────────────────────────
class BiLSTMAttentionScorer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bilstm     = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                                   bidirectional=True, num_layers=2, dropout=0.3)
        self.attn_w     = nn.Linear(hidden_dim * 2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, x):
        emb  = self.embedding(x)
        out, _ = self.bilstm(emb)
        attn = torch.softmax(self.attn_w(out), dim=1)
        ctx  = (attn * out).sum(dim=1)
        return self.classifier(ctx).squeeze(1)


# ── Load checkpoint ──────────────────────────────────────────────────────────
ck1 = torch.load(f'{MODELS_DIR}/bilstm_summarizer.pt', map_location=device)

bilstm_model = BiLSTMAttentionScorer(**ck1['cfg']).to(device)
bilstm_model.load_state_dict(ck1['state'])
bilstm_model.eval()

bilstm_vocab = CharVocab()
bilstm_vocab.char2idx = ck1['vocab']

print('BiLSTM+Attention model loaded.')

BiLSTM+Attention model loaded.


In [ ]:
class CNNSentenceScorer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64,
                 num_filters=128, kernel_sizes=None):
        super().__init__()
        if kernel_sizes is None:
            kernel_sizes = [2, 3, 4]
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs     = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes
        ])
        self.classifier = nn.Sequential(
            nn.Linear(num_filters * len(kernel_sizes), 128),
            nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 1), nn.Sigmoid()
        )

    def forward(self, x):
        emb    = self.embedding(x).permute(0, 2, 1)
        pooled = [torch.relu(c(emb)).max(dim=2).values for c in self.convs]
        return self.classifier(torch.cat(pooled, dim=1)).squeeze(1)


ck2 = torch.load(f'{MODELS_DIR}/cnn_summarizer.pt', map_location=device)

cnn_model = CNNSentenceScorer(**ck2['cfg']).to(device)
cnn_model.load_state_dict(ck2['state'])
cnn_model.eval()

# CNN shares the same vocabulary as BiLSTM (saved in its checkpoint too)
cnn_vocab = CharVocab()
cnn_vocab.char2idx = ck2['vocab']

print('CNN model loaded.')

CNN model loaded.


In [ ]:
!pip install sentencepiece

In [ ]:
#!pip uninstall -y transformers tokenizers accelerate
#!pip install transformers==4.38.2 tokenizers==0.15.2 accelerate==0.27.2

In [ ]:
from transformers import BertTokenizer, BertModel

BERT_MODEL_NAME = "sagorsarker/bangla-bert-base"

# ✅ Use BertTokenizer (NOT AutoTokenizer)
bb_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

bb_encoder = BertModel.from_pretrained(BERT_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# ── Classifier head (same architecture as training) ──────────────────────────
class BERTClassifierHead(nn.Module):
    def __init__(self, hidden=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


# ── Load checkpoint ─────────────────────────────────────────────────────────
ck3 = torch.load(f'{MODELS_DIR}/banglabert_head.pt', map_location=device)

BERT_MODEL_NAME = ck3['bert_model']   # 'sagorsarker/bangla-bert-base'

print(f'Loading BanglaBERT encoder: {BERT_MODEL_NAME} ...')

# ✅ FIX: Use BertTokenizer (NOT AutoTokenizer)
bb_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

bb_encoder = BertModel.from_pretrained(BERT_MODEL_NAME).to(device)
bb_encoder.eval()


# ── Load classifier head ────────────────────────────────────────────────────
bb_head = BERTClassifierHead().to(device)
bb_head.load_state_dict(ck3['head_state'])
bb_head.eval()

print('BanglaBERT model loaded successfully.')

Loading BanglaBERT encoder: sagorsarker/bangla-bert-base ...
BanglaBERT model loaded successfully.


In [ ]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)
print(torch.cuda.is_available())

In [ ]:
# ── Generic DL summarizer (BiLSTM / CNN) ─────────────────────────────────────
def dl_summarize(text, model, vocab, top_n=3, max_len=100):
    """Score each sentence with a char-level DL model and return the top-N."""
    sents = sentence_tokenize_bengali(text)
    if len(sents) <= top_n:
        return ' '.join(sents)
    model.eval()
    with torch.no_grad():
        inp    = torch.tensor(
            [vocab.encode(s, max_len) for s in sents],
            dtype=torch.long
        ).to(device)
        scores = model(inp).cpu().numpy()
    top = sorted(np.argsort(scores)[-top_n:])
    return ' '.join([sents[i] for i in top])


# ── BanglaBERT CLS embeddings ────────────────────────────────────────────────
def get_cls_embeddings(sentences, tokenizer, encoder,
                       batch_size=16, max_len=128):
    all_embs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        enc   = tokenizer(batch, padding=True, truncation=True,
                          max_length=max_len, return_tensors='pt').to(device)
        with torch.no_grad():
            out = encoder(**enc)
        all_embs.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(all_embs, dim=0)


def banglabert_summarize(text, encoder, tokenizer, head, top_n=3):
    sents = sentence_tokenize_bengali(text)
    if len(sents) <= top_n:
        return ' '.join(sents)
    embs = get_cls_embeddings(sents, tokenizer, encoder)
    head.eval()
    with torch.no_grad():
        scores = head(embs.to(device)).cpu().numpy()
    top = sorted(np.argsort(scores)[-top_n:])
    return ' '.join([sents[i] for i in top])


# ── Convenience wrapper — run ALL models on one text ─────────────────────────
def summarize_all(text, top_n=3):
    """
    Run all 5 models on `text` and return a dict of summaries.
    Set top_n to control how many sentences are extracted.
    """
    return {
        'TF-IDF'      : tfidf_model.summarize(text),
        'TextRank'    : textrank_model.summarize(text),
        'BiLSTM+Attn' : dl_summarize(text, bilstm_model, bilstm_vocab, top_n),
        'CNN'         : dl_summarize(text, cnn_model,    cnn_vocab,    top_n),
        'BanglaBERT'  : banglabert_summarize(text, bb_encoder, bb_tokenizer, bb_head, top_n),
    }


print('All inference functions ready.')

All inference functions ready.


In [ ]:
# ✏️  Paste any Bengali text here
my_text = """
বাংলাদেশের রাজধানী ঢাকা বিশ্বের অন্যতম ঘনবসতিপূর্ণ শহর।
এই শহরে প্রতিদিন লক্ষ লক্ষ মানুষ বিভিন্ন কাজে যোগ দেন।
ঢাকার ঐতিহাসিক স্থানগুলোর মধ্যে লালবাগ কেল্লা ও আহসান মঞ্জিল উল্লেখযোগ্য।
শহরের যানজট একটি প্রধান সমস্যা যা সরকার সমাধানের চেষ্টা করছে।
ঢাকার অর্থনীতি মূলত গার্মেন্টস শিল্প ও বিভিন্ন সেবা খাতের উপর নির্ভরশীল।
সাম্প্রতিক বছরগুলোতে শহরে তথ্যপ্রযুক্তি খাতেও ব্যাপক উন্নতি হয়েছে।
"""

results = summarize_all(my_text, top_n=3)

print('=' * 60)
print('ORIGINAL TEXT (first 300 chars):')
print(my_text[:300].strip())
print('=' * 60)

for model_name, summary in results.items():
    print(f'\n🔹 {model_name}:')
    print(summary)

ORIGINAL TEXT (first 300 chars):
বাংলাদেশের রাজধানী ঢাকা বিশ্বের অন্যতম ঘনবসতিপূর্ণ শহর।
এই শহরে প্রতিদিন লক্ষ লক্ষ মানুষ বিভিন্ন কাজে যোগ দেন।
ঢাকার ঐতিহাসিক স্থানগুলোর মধ্যে লালবাগ কেল্লা ও আহসান মঞ্জিল উল্লেখযোগ্য।
শহরের যানজট একটি প্রধান সমস্যা যা সরকার সমাধানের চেষ্টা করছে।
ঢাকার অর্থনীতি মূলত গার্মেন্টস শিল্প ও বিভিন্ন সেবা

🔹 TF-IDF:
ঢাকার ঐতিহাসিক স্থানগুলোর মধ্যে লালবাগ কেল্লা ও আহসান মঞ্জিল উল্লেখযোগ্য ঢাকার অর্থনীতি মূলত গার্মেন্টস শিল্প ও বিভিন্ন সেবা খাতের উপর নির্ভরশীল সাম্প্রতিক বছরগুলোতে শহরে তথ্যপ্রযুক্তি খাতেও ব্যাপক উন্নতি হয়েছে

🔹 TextRank:
এই শহরে প্রতিদিন লক্ষ লক্ষ মানুষ বিভিন্ন কাজে যোগ দেন শহরের যানজট একটি প্রধান সমস্যা যা সরকার সমাধানের চেষ্টা করছে সাম্প্রতিক বছরগুলোতে শহরে তথ্যপ্রযুক্তি খাতেও ব্যাপক উন্নতি হয়েছে

🔹 BiLSTM+Attn:
বাংলাদেশের রাজধানী ঢাকা বিশ্বের অন্যতম ঘনবসতিপূর্ণ শহর এই শহরে প্রতিদিন লক্ষ লক্ষ মানুষ বিভিন্ন কাজে যোগ দেন শহরের যানজট একটি প্রধান সমস্যা যা সরকার সমাধানের চেষ্টা করছে

🔹 CNN:
বাংলাদেশের রাজধানী ঢাকা বিশ্বের অন্যতম ঘনবসতিপূর্ণ শহর ঢ

In [ ]:
# Cell 5 ─ Text preprocessing + OCR + file extraction

def normalize_bengali(text):
    text = unicodedata.normalize('NFC', str(text))
    text = text.replace('\u200c','').replace('\u200d','')
    text = re.sub(r'[\u0964\u0965]+', '\u0964', text)
    text = re.sub(r'[^\u0980-\u09FF\u09E6-\u09EF0-9\s\u0964,.?!\-()]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def extract_text_from_image(pil_img):
    """OCR a PIL image — returns Bengali + English text."""
    try:
        return pytesseract.image_to_string(pil_img, lang='ben+eng', config='--psm 3').strip()
    except Exception:
        return pytesseract.image_to_string(pil_img, config='--psm 3').strip()

def extract_text_from_file(raw_bytes, filename):
    """Extract text from PDF / DOCX / TXT bytes."""
    ext = Path(filename).suffix.lower()
    import io
    if ext == '.pdf':
        parts = []
        with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
            for page in pdf.pages:
                t = page.extract_text()
                if t: parts.append(t)
        return '\n'.join(parts)
    elif ext in ('.docx', '.doc'):
        doc = python_docx.Document(io.BytesIO(raw_bytes))
        return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        return raw_bytes.decode('utf-8', errors='ignore')

print('✅ Text utilities ready')


✅ Text utilities ready


In [ ]:
import cv2
import numpy as np
import pytesseract

def extract_text_from_image(pil_img):
    try:
        # Convert PIL → OpenCV
        img = np.array(pil_img)

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # Denoise
        gray = cv2.medianBlur(gray, 3)

        # Threshold (VERY IMPORTANT for Bengali)
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

        # OCR with better config
        text = pytesseract.image_to_string(
            thresh,
            lang='ben',                     # 🔥 only Bengali
            config='--oem 3 --psm 6'        # 🔥 better layout mode
        )

        return normalize_bengali(text)

    except Exception as e:
        return f"OCR Error: {e}"

In [ ]:
#!apt-get install -y tesseract-ocr
#!pip install pytesseract pdfplumber python-docx

In [ ]:
# Cell 4 ─ Imports
import re, os, pickle, unicodedata, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import pytesseract
import pdfplumber
import docx as python_docx
from transformers import (
    AutoTokenizer, AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM, pipeline as hf_pipeline,
)

RuntimeError: Failed to import transformers.pipelines because of the following error (look up to see its traceback):
cannot import name 'is_offline_mode' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

Using device: cuda


In [ ]:
# Cell 6 ─ PyTorch model class definitions

MAX_CTX = 300
MAX_Q   = 64

# ── 1. Attention (ADD version) ───────────────────────────────
class BiLSTM_Attn_QA(nn.Module):
    def __init__(self, V, E, H, emb):
        super().__init__()
        self.name = 'BiLSTM+Attention (ADD)'

        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.emb.weight = nn.Parameter(emb, requires_grad=True)

        self.ctx_lstm = nn.LSTM(E, H, batch_first=True,
                               bidirectional=True, num_layers=2, dropout=0.3)

        self.q_lstm = nn.LSTM(E, H, batch_first=True,
                             bidirectional=True, num_layers=2, dropout=0.3)

        self.drop = nn.Dropout(0.3)

        # H*2
        self.sf = nn.Linear(H*2, 1)
        self.ef = nn.Linear(H*2, 1)

    def forward(self, ctx, q, **kw):
        co, _ = self.ctx_lstm(self.drop(self.emb(ctx)))
        qo, _ = self.q_lstm(self.drop(self.emb(q)))

        attn = torch.softmax(torch.bmm(co, qo.transpose(1,2)), dim=-1)

        comb = self.drop(co + torch.bmm(attn, qo))

        return self.sf(comb).squeeze(-1), self.ef(comb).squeeze(-1)


# ── 2. Attention (CONCAT version) ─────────────────────────────
class BiLSTM_Attn_QA_Concat(nn.Module):
    def __init__(self, V, E, H, emb):
        super().__init__()
        self.name = 'BiLSTM+Attention (CONCAT)'

        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.emb.weight = nn.Parameter(emb, requires_grad=True)

        self.ctx_lstm = nn.LSTM(E, H, batch_first=True,
                               bidirectional=True, num_layers=2, dropout=0.3)

        self.q_lstm = nn.LSTM(E, H, batch_first=True,
                             bidirectional=True, num_layers=2, dropout=0.3)

        self.aw = nn.Linear(H*2, H*2, bias=False)

        self.drop = nn.Dropout(0.3)

        # H*4
        self.sf = nn.Linear(H*4, 1)
        self.ef = nn.Linear(H*4, 1)

    def forward(self, ctx, q, **kw):
        co, _ = self.ctx_lstm(self.drop(self.emb(ctx)))
        qo, _ = self.q_lstm(self.drop(self.emb(q)))

        attn = torch.softmax(torch.bmm(co, self.aw(qo).transpose(1,2)), dim=-1)

        comb = self.drop(torch.cat([co, torch.bmm(attn, qo)], dim=-1))

        return self.sf(comb).squeeze(-1), self.ef(comb).squeeze(-1)


# ── 3. CNN + BiLSTM ───────────────────────────────────────────
class CNN_BiLSTM_QA(nn.Module):
    def __init__(self, V, E, H, emb):
        super().__init__()
        self.name = 'CNN+BiLSTM'

        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.emb.weight = nn.Parameter(emb, requires_grad=True)

        self.convs = nn.ModuleList([
            nn.Conv1d(E, H, k, padding=k//2) for k in [3,5,7]
        ])

        self.ctx_lstm = nn.LSTM(H*3, H, batch_first=True,
                               bidirectional=True, num_layers=2, dropout=0.3)

        self.q_lstm = nn.LSTM(E, H, batch_first=True,
                             bidirectional=True, num_layers=2, dropout=0.3)

        self.aw = nn.Linear(H*2, H*2, bias=False)

        self.drop = nn.Dropout(0.3)

        self.sf = nn.Linear(H*4, 1)
        self.ef = nn.Linear(H*4, 1)

    def forward(self, ctx, q, **kw):
        T = ctx.size(1)

        ex = self.drop(self.emb(ctx)).transpose(1,2)

        co = torch.cat([
            torch.relu(c(ex))[:, :, :T].transpose(1,2)
            for c in self.convs
        ], dim=-1)

        co, _ = self.ctx_lstm(co)

        qo, _ = self.q_lstm(self.drop(self.emb(q)))

        attn = torch.softmax(torch.bmm(co, self.aw(qo).transpose(1,2)), dim=-1)

        comb = self.drop(torch.cat([co, torch.bmm(attn, qo)], dim=-1))

        return self.sf(comb).squeeze(-1), self.ef(comb).squeeze(-1)


print('✅ Model classes defined')

✅ Model classes defined


In [ ]:
# Cell 7 ─ Load all models
DRIVE_DIR = '/content/drive/MyDrive/NLP_Project_Models'
D = DRIVE_DIR

print('Loading vocab + embeddings...')
with open(f'{D}/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

V = len(vocab)

embed_state = torch.load(f'{D}/embedding.pt', map_location='cpu')

embed_tensor = list(embed_state.values())[0] if isinstance(embed_state, dict) else embed_state

EMBED_DIM = embed_tensor.shape[1]
HIDDEN_DIM = 256

print(f'  Vocab: {V}  |  Embed dim: {EMBED_DIM}')


# ── LOAD MODELS ───────────────────────────────
print('Loading PyTorch QA models...')

# bilstm.pt → ADD attention
m1 = BiLSTM_Attn_QA(V, EMBED_DIM, HIDDEN_DIM, embed_tensor).to(DEVICE)
m1.load_state_dict(torch.load(f'{D}/bilstm.pt', map_location=DEVICE))
m1.eval()

# bilstm_attn.pt → CONCAT attention
m2 = BiLSTM_Attn_QA_Concat(V, EMBED_DIM, HIDDEN_DIM, embed_tensor).to(DEVICE)
m2.load_state_dict(torch.load(f'{D}/bilstm_attn.pt', map_location=DEVICE))
m2.eval()

# cnn_bilstm.pt → CNN model
m3 = CNN_BiLSTM_QA(V, EMBED_DIM, HIDDEN_DIM, embed_tensor).to(DEVICE)
m3.load_state_dict(torch.load(f'{D}/cnn_bilstm.pt', map_location=DEVICE))
m3.eval()

print('✅ ALL QA MODELS LOADED SUCCESSFULLY')

Loading vocab + embeddings...
  Vocab: 35574  |  Embed dim: 128
Loading PyTorch QA models...
✅ ALL QA MODELS LOADED SUCCESSFULLY


In [ ]:
# Cell 8 ─ PyTorch QA inference helper

def answer_pytorch(mdl, context, question):
    device = next(mdl.parameters()).device
    ct = normalize_bengali(context).split()
    qt = normalize_bengali(question).split()
    ci = torch.tensor([[vocab.get(t,1) for t in ct[:MAX_CTX]]]).to(device)
    qi = torch.tensor([[vocab.get(t,1) for t in qt[:MAX_Q]]]).to(device)
    mdl.eval()
    with torch.no_grad():
        sl, el = mdl(ci, qi)
    si = sl[0].argmax().item()
    ei = max(el[0].argmax().item(), si)
    ans = ' '.join(ct[si:ei+1]).strip()
    return ans or '(উত্তর পাওয়া যায়নি)'

print('✅ QA inference helper ready')


✅ QA inference helper ready


In [ ]:
# ── S-Cell 3: Bengali text utilities (shared) ────────────────────────────────

def clean_bengali(text):
    """Light cleaning — keep Bengali script + spaces + punctuation."""
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\u200c','').replace('\u200d','')
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def sentence_split_bengali(text):
    """Split Bengali text into sentences on ।  ? ! ."""
    text = clean_bengali(text)
    sents = re.split(r'[।?!\.]+', text)
    return [s.strip() for s in sents if s.strip() and len(s.strip().split()) >= 3]


# ── Document / Image loaders ─────────────────────────────────────────────────
def extract_text_from_pdf(raw_bytes):
    if not HAS_FITZ: raise RuntimeError('PyMuPDF not installed')
    doc = fitz.open(stream=raw_bytes, filetype='pdf')
    return ' '.join(page.get_text() for page in doc).strip()

def extract_text_from_file(raw_bytes, filename='file.txt'):
    ext = os.path.splitext(filename)[1].lower()
    if ext == '.pdf':
        return extract_text_from_pdf(raw_bytes)
    elif ext == '.docx':
        from docx import Document
        doc = Document(io.BytesIO(raw_bytes))
        return '\n'.join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        return raw_bytes.decode('utf-8', errors='ignore')

def extract_text_from_image(pil_img):
    try:
        return pytesseract.image_to_string(pil_img, lang='ben+eng', config='--oem 3 --psm 3').strip()
    except Exception:
        return pytesseract.image_to_string(pil_img, config='--oem 3 --psm 3').strip()

print('✅ Utilities ready')

✅ Utilities ready


In [ ]:
import torch
import torch.nn as nn

class BiLSTMSummarizer(nn.Module):
    """
    BiLSTM Extractive Summarizer for Bengali text.

    Architecture:
      Embedding → BiLSTM → Mean Pooling → Linear → Sentence Score
    """

    def __init__(self, vocab_size, embed_dim=64, hidden_size=128,
                 num_layers=1, dropout=0.3, padding_idx=0):
        super(BiLSTMSummarizer, self).__init__()

        self.embed_dim   = embed_dim
        self.hidden_size = hidden_size

        # 🔹 Embedding Layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=padding_idx
        )

        # 🔹 BiLSTM Layer
        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        # 🔹 Dropout
        self.dropout = nn.Dropout(dropout)

        # 🔹 Sentence scoring layer
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        """
        Args:
            x : (batch, seq_len)

        Returns:
            scores : (batch,)
        """

        # Step 1: Embedding
        embedded = self.dropout(self.embedding(x))

        # Step 2: BiLSTM
        output, _ = self.bilstm(embedded)

        # Step 3: Mask padding
        mask = (x != 0).unsqueeze(-1).float()
        masked_output = output * mask

        lengths = mask.sum(dim=1).clamp(min=1)

        # Step 4: Mean pooling
        sentence_vec = masked_output.sum(dim=1) / lengths

        # Step 5: Score
        score = self.fc(sentence_vec).squeeze(-1)

        return score

In [ ]:
EMBED_DIM   = 64
HIDDEN_SIZE = 128
MAX_LEN     = 30

model = BiLSTMSummarizer(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE
).to(DEVICE)

print("✅ BiLSTM Summarizer Built!")
print(model)

✅ BiLSTM Summarizer Built!
BiLSTMSummarizer(
  (embedding): Embedding(35574, 64, padding_idx=0)
  (bilstm): LSTM(64, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [ ]:
import re

def summarize_with_bilstm(text, model, vocab, top_n=3):
    sentences = re.split(r'[।.!?]+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]

    if len(sentences) <= top_n:
        return text

    scores = []

    model.eval()
    with torch.no_grad():
        for sent in sentences:
            tokens = sent.split()

            idx = [vocab.get(w, 1) for w in tokens[:MAX_LEN]]
            tensor = torch.tensor([idx]).to(DEVICE)

            score = model(tensor).item()
            scores.append(score)

    # Select top sentences
    top_idx = sorted(
        sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]
    )

    summary = ' ।\n'.join(sentences[i] for i in top_idx) + ' ।'
    return summary

In [ ]:
text = "বাংলাদেশ একটি স্বাধীন দেশ। এর রাজধানী ঢাকা। এটি একটি উন্নয়নশীল দেশ। কৃষি ও শিল্পে উন্নতি হয়েছে।"

summary = summarize_with_bilstm(text, model, vocab)

print(summary)

বাংলাদেশ একটি স্বাধীন দেশ ।
এর রাজধানী ঢাকা ।
এটি একটি উন্নয়নশীল দেশ ।


In [ ]:
def bilstm_sum_wrapper(text):
    return summarize_with_bilstm(text, m1, vocab)   # or m2

In [ ]:
import os

tok_file = f"{D}/mt5_summarizer/tokenizer.json"

if os.path.exists(tok_file):
    os.remove(tok_file)
    print("Deleted broken tokenizer.json ✅")

Deleted broken tokenizer.json ✅


In [ ]:
!pip install transformers==4.38.2 tokenizers==0.15.2 sentencepiece

  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.38.2-py3-none-any.whl (8.5 MB)
Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.9.0
    Uninstalling huggingface_hub-1.9.0:
      Successfully uninstalled huggingface_hub-1.9.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.save_pretrained(f"{D}/mt5_summarizer")
model.save_pretrained(f"{D}/mt5_summarizer")

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_path = f"{D}/mt5_summarizer"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=False   # 🔥 IMPORTANT
)

model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)

print("Loaded successfully ✅")

Loaded successfully ✅


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# ✅ FIX: force slow tokenizer
mt5_tok = AutoTokenizer.from_pretrained(
    f'{D}/mt5_summarizer',
    use_fast=False
)

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    f'{D}/mt5_summarizer'
).to(DEVICE)

mt5_model.eval()


def mt5_sum(text):
    inputs = mt5_tok(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding='max_length'
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = mt5_model.generate(
            **inputs,
            max_new_tokens=150,
            num_beams=4,          # 🔥 better summaries
            early_stopping=True
        )

    return mt5_tok.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
# Install Whisper
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai-whisper', 'ffmpeg-python'], check=False)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=False)
print('✅ Whisper installed')

In [ ]:
import whisper
import tempfile, os
import numpy as np

# Load once — 'base' is fast, 'medium' is more accurate for Bengali
whisper_model = whisper.load_model('medium')
print(f'✅ Whisper loaded: {whisper_model.dims}')

In [ ]:
def transcribe_audio(audio_input):
    """
    Takes audio from Gradio microphone (numpy array + sample rate)
    or an uploaded audio file path.
    Returns transcribed Bengali text.
    """
    if audio_input is None:
        return '', '⚠️ No audio received.'

    try:
        # Gradio mic returns (sample_rate, numpy_array)
        if isinstance(audio_input, tuple):
            sample_rate, audio_data = audio_input

            # Convert to float32 mono
            if audio_data.ndim == 2:
                audio_data = audio_data.mean(axis=1)   # stereo → mono
            audio_data = audio_data.astype(np.float32)

            # Normalize if int16
            if audio_data.max() > 1.0:
                audio_data = audio_data / 32768.0

            # Save to temp wav so whisper can read it
            import soundfile as sf
            with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                tmp_path = tmp.name
            sf.write(tmp_path, audio_data, sample_rate)

        else:
            # Uploaded file path
            tmp_path = audio_input

        # Transcribe — force Bengali language
        result = whisper_model.transcribe(
            tmp_path,
            language='bn',          # Bengali
            task='transcribe',      # keep in Bengali (use 'translate' for English)
            fp16=torch.cuda.is_available(),
        )
        text = result['text'].strip()

        # Cleanup temp file
        if isinstance(audio_input, tuple) and os.path.exists(tmp_path):
            os.remove(tmp_path)

        if not text:
            return '', '⚠️ No speech detected.'

        return text, f'✅ Transcribed: {len(text)} characters'

    except Exception as e:
        return '', f'❌ Transcription error: {e}'


def speech_then_summarize(audio_input, sum_model_choice):
    """Transcribe audio → store in doc_state → summarize."""
    text, status = transcribe_audio(audio_input)
    if not text:
        return '', status, ''
    doc_state['text'] = text
    summary = run_summarize(text, sum_model_choice)
    return text, status, summary


def speech_then_qa(audio_input, question, qa_model_choice):
    """
    Two modes:
    1. audio_input  = the CONTEXT spoken aloud → transcribe, then answer question
    2. question     = the QUESTION spoken via separate mic (handled in UI)
    """
    context_text, status = transcribe_audio(audio_input)
    if not context_text:
        return '', status, ''
    doc_state['text'] = context_text
    answer = run_qa(context_text, question, qa_model_choice)
    return context_text, status, answer


def speech_question(audio_input):
    """Transcribe a spoken question only — text goes into question box."""
    text, status = transcribe_audio(audio_input)
    return text, status


print('✅ Speech functions ready')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile'], check=False)
print('✅ soundfile installed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q transformers torchaudio librosa soundfile

import torch
import librosa
from transformers import AutoProcessor, AutoModelForCTC
from google.colab import files

# 📤 Upload file
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]

# 🔁 Load audio (16kHz mono)
speech, sr = librosa.load(audio_path, sr=16000, mono=True)

# 🔥 Load model
model_name = "sazzadul/Shrutimala_Bangla_ASR"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForCTC.from_pretrained(model_name)

# 🎯 Process input
inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)

print("Processor output keys:", inputs.keys())  # 👈 debug

# ✅ Handle BOTH cases safely
if "input_values" in inputs:
    model_input = inputs["input_values"]
elif "input_features" in inputs:
    model_input = inputs["input_features"]
else:
    raise ValueError("No valid input found from processor!")

# 🚀 Inference
with torch.no_grad():
    logits = model(model_input).logits

# 📝 Decode
predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)[0]

print("📝 Transcription:")
print(transcription)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 41.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

special_tokens_map.json:   0%|          | 0.00/102 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.42G [00:00<?, ?B/s]

Processor output keys: dict_keys(['input_features', 'attention_mask'])
📝 Transcription:



In [ ]:
save_path = "/content/drive/MyDrive/bangla_asr_model"

processor.save_pretrained(save_path)
model.save_pretrained(save_path)

print("✅ Model saved to Drive:", save_path)

✅ Model saved to Drive: /content/drive/MyDrive/bangla_asr_model


In [ ]:
from transformers import AutoProcessor, AutoModelForCTC

load_path = "/content/drive/MyDrive/bangla_asr_model"

processor = AutoProcessor.from_pretrained(load_path)
model = AutoModelForCTC.from_pretrained(load_path)

print("✅ Model loaded from Drive")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


TypeError: Wav2Vec2BertProcessor.__init__() got multiple values for argument 'feature_extractor'

In [ ]:
#!pip install --upgrade --force-reinstall transformers==4.41.2 --no-cache-dir

In [ ]:
import sys
sys.modules.pop("transformers", None)

In [ ]:
from transformers import pipeline
import torch

# ── BanglaBERT / XLM-R QA ─────────────────────────
bangla_pipe = pipeline(
    'question-answering',
    model='deepset/xlm-roberta-large-squad2',
    tokenizer='deepset/xlm-roberta-large-squad2',
    device=0 if torch.cuda.is_available() else -1
)
print('✅ BanglaBERT QA loaded')

# ── XLM-RoBERTa QA ───────────────────────────────
xlmr_pipe = pipeline(
    'question-answering',
    model='deepset/xlm-roberta-base-squad2',
    tokenizer='deepset/xlm-roberta-base-squad2',
    device=0 if torch.cuda.is_available() else -1
)
print('✅ XLM-RoBERTa loaded')

# ── Zero-shot QA ─────────────────────────────────
qa_zs = pipeline(
    'question-answering',
    model='deepset/roberta-base-squad2',
    device=0 if torch.cuda.is_available() else -1
)
print('✅ Zero-shot QA loaded')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at deepset/xlm-roberta-large-squad2 were not used when initializing XLMRobertaForQuestionAnswering: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

✅ Zero-shot QA loaded


In [ ]:
import os
import io
doc_state = {'text': ''}

# Dummy implementations (replace with your real ones)

def ui_load_file(file_obj):
    if file_obj is None:
        return "", "⚠️ No file"
    return "Sample extracted text", "✅ File loaded"

def ui_load_image(img):
    return "Sample OCR text", "✅ OCR done"

def run_summarize(text, model):
    return f"Summary of: {text[:50]}..."

def run_qa(context, question, model):
    return f"Answer to '{question}' from context"

def ui_summarize_loaded(model):
    return run_summarize(doc_state.get('text', ''), model)

def ui_qa_loaded(question, model):
    return run_qa(doc_state.get('text', ''), question, model)

In [ ]:
# ============================================================
# Cell 9 ─ Full Gradio UI  (Speech + Summarization + QA)
# ============================================================
# Summarization models (no mT5 except mT5-XLSum which stays):
#   TF-IDF, TextRank, BiLSTM+Attention, CNN, BanglaBERT
# Speech: sazzadul/Shrutimala_Bangla_ASR  (CTC model)
# ============================================================

# ── 0. Extra installs (run once) ──────────────────────────────────────────
import subprocess, sys
import io
def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for _p in ['gradio', 'librosa', 'soundfile', 'torchaudio']:
    _install(_p)

# ── 1. Imports ────────────────────────────────────────────────────────────
import os, re, pickle, tempfile, numpy as np, torch
import torch.nn as nn
import librosa, gradio as gr
from transformers import AutoProcessor, AutoModelForCTC
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ────────────────────────────────────────────────────────────────────────────
# ── 2. Text utilities ─────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
def normalize_bengali(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def sentence_tokenize_bengali(text: str):
    sents = re.split(r'[।.!?]+', text)
    return [s.strip() for s in sents if len(s.strip()) > 10]

# ────────────────────────────────────────────────────────────────────────────
# ── 3. Model class definitions (needed for pickle / torch.load) ───────────
# ────────────────────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

class TFIDFSummarizer:
    def __init__(self, top_n=3):
        self.top_n = top_n
        self.vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
    def summarize(self, text):
        sents = sentence_tokenize_bengali(text)
        if len(sents) <= self.top_n:
            return ' '.join(sents)
        mat    = self.vectorizer.transform(sents)
        scores = np.array(mat.sum(axis=1)).flatten()
        top    = sorted(np.argsort(scores)[-self.top_n:])
        return ' '.join([sents[i] for i in top])
    @staticmethod
    def load(path):
        with open(path, 'rb') as f:
            return pickle.load(f)

class TextRankSummarizer:
    def __init__(self, top_n=3):
        self.top_n = top_n
        self.vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
    def summarize(self, text):
        sents = sentence_tokenize_bengali(text)
        if len(sents) <= self.top_n:
            return ' '.join(sents)
        mat = self.vectorizer.transform(sents).toarray()
        sim = cosine_similarity(mat)
        np.fill_diagonal(sim, 0)
        G = nx.from_numpy_array(sim)
        try:
            scores = nx.pagerank(G, max_iter=300)
        except nx.PowerIterationFailedConvergence:
            scores = {i: 1/len(sents) for i in range(len(sents))}
        ranked = sorted(scores, key=scores.get, reverse=True)[:self.top_n]
        return ' '.join([sents[i] for i in sorted(ranked)])
    @staticmethod
    def load(path):
        with open(path, 'rb') as f:
            return pickle.load(f)

class CharVocab:
    def __init__(self, max_vocab=5000):
        self.max_vocab = max_vocab
        self.char2idx  = {'<PAD>': 0, '<UNK>': 1}
    def encode(self, text, max_len=100):
        ids = [self.char2idx.get(c, 1) for c in text[:max_len]]
        ids += [0] * (max_len - len(ids))
        return ids

class BiLSTMAttentionScorer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bilstm     = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                                   bidirectional=True, num_layers=2, dropout=0.3)
        self.attn_w     = nn.Linear(hidden_dim * 2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        emb  = self.embedding(x)
        out, _ = self.bilstm(emb)
        attn = torch.softmax(self.attn_w(out), dim=1)
        ctx  = (attn * out).sum(dim=1)
        return self.classifier(ctx).squeeze(1)

class CNNSentenceScorer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64,
                 num_filters=128, kernel_sizes=None):
        super().__init__()
        if kernel_sizes is None:
            kernel_sizes = [2, 3, 4]
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs     = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes
        ])
        self.classifier = nn.Sequential(
            nn.Linear(num_filters * len(kernel_sizes), 128),
            nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, x):
        emb    = self.embedding(x).permute(0, 2, 1)
        pooled = [torch.relu(c(emb)).max(dim=2).values for c in self.convs]
        return self.classifier(torch.cat(pooled, dim=1)).squeeze(1)

class BERTClassifierHead(nn.Module):
    def __init__(self, hidden=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

# ────────────────────────────────────────────────────────────────────────────
# ── 4. Load saved models from Drive ──────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
# ✏️  Update this path if your folder is elsewhere in Drive
MODELS_DIR = '/content/drive/MyDrive/NLP_Project_Models/summarizer'

print('Loading models …')

# TF-IDF
tfidf_model    = TFIDFSummarizer.load(f'{MODELS_DIR}/tfidf_summarizer.pkl')

# TextRank
textrank_model = TextRankSummarizer.load(f'{MODELS_DIR}/textrank_summarizer.pkl')

# BiLSTM
ck1            = torch.load(f'{MODELS_DIR}/bilstm_summarizer.pt', map_location=device)
bilstm_model   = BiLSTMAttentionScorer(**ck1['cfg']).to(device)
bilstm_model.load_state_dict(ck1['state'])
bilstm_model.eval()
bilstm_vocab   = CharVocab(); bilstm_vocab.char2idx = ck1['vocab']

# CNN
ck2            = torch.load(f'{MODELS_DIR}/cnn_summarizer.pt', map_location=device)
cnn_model      = CNNSentenceScorer(**ck2['cfg']).to(device)
cnn_model.load_state_dict(ck2['state'])
cnn_model.eval()
cnn_vocab      = CharVocab()
cnn_vocab.char2idx = ck2.get('vocab', ck1['vocab'])   # fallback to bilstm vocab

# BanglaBERT encoder + head
BB_NAME        = 'sagorsarker/bangla-bert-base'
print(f'  Loading BanglaBERT encoder ({BB_NAME}) …')
bb_tokenizer   = AutoTokenizer.from_pretrained(BB_NAME)
bb_encoder     = AutoModel.from_pretrained(BB_NAME).to(device)
bb_encoder.eval()
ck3            = torch.load(f'{MODELS_DIR}/banglabert_head.pt', map_location=device)
bb_head        = BERTClassifierHead().to(device)
bb_head.load_state_dict(ck3['head_state'])
bb_head.eval()

# Speech ASR model (lazy-loaded on first use to save memory)
_asr_processor = None
_asr_model     = None
ASR_MODEL_NAME = 'sazzadul/Shrutimala_Bangla_ASR'

def _load_asr():
    global _asr_processor, _asr_model
    if _asr_processor is None:
        print(f'  Loading ASR model ({ASR_MODEL_NAME}) …')
        _asr_processor = AutoProcessor.from_pretrained(ASR_MODEL_NAME)
        _asr_model     = AutoModelForCTC.from_pretrained(ASR_MODEL_NAME)
        _asr_model.eval()
    return _asr_processor, _asr_model

print('All models loaded ✅')

# ────────────────────────────────────────────────────────────────────────────
# ── 5. Inference helpers ──────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
def _get_cls_embeddings(sentences, batch_size=16, max_len=128):
    all_embs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        enc   = bb_tokenizer(batch, padding=True, truncation=True,
                             max_length=max_len, return_tensors='pt').to(device)
        with torch.no_grad():
            out = bb_encoder(**enc)
        all_embs.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(all_embs, dim=0)

def _dl_summarize(text, model, vocab, top_n=3, max_len=100):
    sents = sentence_tokenize_bengali(text)
    if len(sents) <= top_n:
        return ' '.join(sents)
    model.eval()
    with torch.no_grad():
        inp    = torch.tensor(
            [vocab.encode(s, max_len) for s in sents], dtype=torch.long
        ).to(device)
        scores = model(inp).cpu().numpy()
    top = sorted(np.argsort(scores)[-top_n:])
    return ' '.join([sents[i] for i in top])

# ── Public summarize functions (fn(text) -> str) ──────────────────────────
def tfidf_sum(text):
    return tfidf_model.summarize(text)

def textrank_sum(text):
    return textrank_model.summarize(text)

def bilstm_sum(text):
    return _dl_summarize(text, bilstm_model, bilstm_vocab)

def cnn_sum(text):
    return _dl_summarize(text, cnn_model, cnn_vocab)

def banglabert_sum(text, top_n=3):
    sents = sentence_tokenize_bengali(text)
    if len(sents) <= top_n:
        return ' '.join(sents)
    embs = _get_cls_embeddings(sents)
    bb_head.eval()
    with torch.no_grad():
        scores = bb_head(embs.to(device)).cpu().numpy()
    top = sorted(np.argsort(scores)[-top_n:])
    return ' '.join([sents[i] for i in top])

# ── ASR transcription ─────────────────────────────────────────────────────
def transcribe_audio(audio_input):
    """
    audio_input: tuple (sample_rate, np.ndarray) from gr.Audio
    Returns: Bengali transcription string
    """
    if audio_input is None:
        return '', '⚠️ No audio provided.'
    try:
        proc, mdl = _load_asr()
        sr, audio_arr = audio_input
        # Convert to float32 mono
        audio_arr = audio_arr.astype(np.float32)
        if audio_arr.ndim > 1:
            audio_arr = audio_arr.mean(axis=1)
        # Normalize
        if audio_arr.max() > 1.0:
            audio_arr = audio_arr / 32768.0
        # Resample to 16 kHz if needed
        if sr != 16000:
            audio_arr = librosa.resample(audio_arr, orig_sr=sr, target_sr=16000)
        inputs = proc(audio_arr, sampling_rate=16000,
                      return_tensors='pt', padding=True)
        model_input = (inputs['input_values']
                       if 'input_values' in inputs
                       else inputs['input_features'])
        with torch.no_grad():
            logits = mdl(model_input).logits
        predicted_ids  = torch.argmax(logits, dim=-1)
        transcription  = proc.batch_decode(predicted_ids)[0]
        return transcription, f'✅ Transcription complete ({len(transcription)} chars)'
    except Exception as e:
        return '', f'❌ ASR Error: {e}'

# ────────────────────────────────────────────────────────────────────────────
# ── 6. Shared document state ──────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
doc_state = {'text': ''}

# ────────────────────────────────────────────────────────────────────────────
# ── 7. Summarization dispatcher ──────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
SUM_MODELS = {
    'TF-IDF (Extractive)'          : tfidf_sum,
    'TextRank (Extractive)'        : textrank_sum,
    'BiLSTM+Attention (Extractive)': bilstm_sum,
    'CNN (Extractive)'             : cnn_sum,
    'BanglaBERT (Extractive) ⭐'   : banglabert_sum,
    'mT5-XLSum (Abstractive)'      : mt5_sum,     # kept unchanged from original
    'All Models (Comparison)'      : None,
}

def run_summarize(text, model_choice):
    text = normalize_bengali(text.strip())
    if not text:
        return '⚠️ Please enter text.'
    if model_choice == 'All Models (Comparison)':
        parts = []
        for name, fn in list(SUM_MODELS.items())[:-1]:
            try:
                parts.append(f'**{name}:**\n{fn(text)}')
            except Exception as e:
                parts.append(f'**{name}:** ❌ {e}')
        return '\n\n---\n\n'.join(parts)
    fn = SUM_MODELS.get(model_choice)
    if fn is None:
        return '⚠️ Model not found.'
    try:
        return fn(text)
    except Exception as e:
        return f'❌ Error: {e}'

# ────────────────────────────────────────────────────────────────────────────
# ── 8. QA dispatcher (unchanged from original) ────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
QA_MODELS = [
    'BanglaBERT (FT) ⭐',
    'XLM-RoBERTa (Base)',
    'Zero-shot',
    'BiLSTM',
    'BiLSTM+Attention',
    'CNN+BiLSTM',
    'All Models (Comparison)',
]

MAX_CTX = 500   # word limit for lightweight DL QA models

def run_qa(context, question, model_choice):
    context  = context.strip()
    question = question.strip()
    if not context:  return '⚠️ Please provide context text.'
    if not question: return '⚠️ Please enter a question.'
    ctx_short = ' '.join(context.split()[:MAX_CTX])

    def _get(name):
        if 'BanglaBERT' in name:
            return bangla_pipe(question=question, context=context)['answer']
        elif 'XLM' in name:
            return xlmr_pipe(question=question, context=context)['answer']
        elif 'Zero' in name:
            return qa_zs(question=question, context=context)['answer']
        elif name == 'BiLSTM':
            return answer_pytorch(m1, ctx_short, question)
        elif name == 'BiLSTM+Attention':
            return answer_pytorch(m2, ctx_short, question)
        elif name == 'CNN+BiLSTM':
            return answer_pytorch(m3, ctx_short, question)
        return ''

    try:
        if model_choice == 'All Models (Comparison)':
            lines = []
            for nm in QA_MODELS[:-1]:
                ans = _get(nm)
                lines.append(f'**{nm}:** {ans or "(No answer found)"}')
            return '\n\n'.join(lines)
        return _get(model_choice) or '(No answer found)'
    except Exception as e:
        return f'❌ Error: {e}'

# ────────────────────────────────────────────────────────────────────────────
# ── 9. File / Image loaders (unchanged) ──────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
def ui_load_file(file_obj):
    if file_obj is None: return '', '⚠️ No file uploaded.'
    try:
        with open(file_obj.name, 'rb') as f:
            raw = f.read()
        fname = os.path.basename(file_obj.name)
        text  = extract_text_from_file(raw, fname)
        if not text.strip(): return '', '⚠️ No text found.'
        doc_state['text'] = text
        preview = text[:1500] + ('...' if len(text) > 1500 else '')
        return preview, f'✅ {fname} ({len(text)} chars)'
    except Exception as e:
        return '', f'❌ {e}'

def ui_load_image(pil_img):
    if pil_img is None:
        return '', '⚠️ No image uploaded.'

    try:
        text = extract_text_from_image(pil_img)
        text = normalize_bengali(text)   # 🔥 IMPORTANT CLEANING

        if not text.strip():
            return '', '⚠️ No text extracted via OCR.'

        doc_state['text'] = text

        preview = text[:1500] + ('...' if len(text) > 1500 else '')
        return preview, f'✅ OCR successful ({len(text)} chars)'

    except Exception as e:
        return '', f'❌ OCR Error: {e}'

# ── UI wrappers for loaded-doc tabs ───────────────────────────────────────
def ui_summarize_loaded(sum_model):
    t = doc_state['text'].strip()
    if not t: return '⚠️ Please upload a document/image first.'
    return run_summarize(t, sum_model)

def ui_qa_loaded(question, qa_model):
    t = doc_state['text'].strip()
    if not t: return '⚠️ Please upload a document/image first.'
    return run_qa(t, question, qa_model)

# ────────────────────────────────────────────────────────────────────────────
# ── 10. Speech tab helpers ────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
_speech_text = {'value': ''}   # holds last transcription for summarize/QA

def ui_transcribe(audio):
    text, status = transcribe_audio(audio)
    _speech_text['value'] = text
    doc_state['text']     = text          # also make available to other tabs
    return text, status

def ui_speech_summarize(text_override, sum_model):
    text = (text_override or _speech_text['value']).strip()
    if not text:
        return '⚠️ Please transcribe audio first (or type text above).'
    return run_summarize(text, sum_model)

def ui_speech_qa(text_override, question, qa_model):
    text = (text_override or _speech_text['value']).strip()
    if not text:
        return '⚠️ Please transcribe audio first (or type context above).'
    return run_qa(text, question, qa_model)

# ────────────────────────────────────────────────────────────────────────────
# ── 11. Build Gradio UI ───────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────
with gr.Blocks(title='Bengali NLP System') as demo:

    gr.HTML("""
    <div style='text-align:center;padding:16px 0'>
      <h1 style='color:#006a4e'>🇧🇩 Bengali NLP System</h1>
      <p style='color:#555'>
        Document QA &amp; Summarization &nbsp;|&nbsp; Image OCR &nbsp;|&nbsp;
        🎙️ Speech-to-Text &nbsp;|&nbsp;
        BanglaBERT · XLM-R · mT5 · BiLSTM · CNN · TF-IDF
      </p>
    </div>
    """)

    # ── Tab 1: Document Upload ─────────────────────────────────────────────
    with gr.Tab('📄 Document'):
        gr.Markdown('Upload a **PDF / TXT / DOCX** file. Text is extracted automatically.')
        with gr.Row():
            with gr.Column(scale=1):
                file_up   = gr.File(label='Upload File (PDF / TXT / DOCX)')
                file_st   = gr.Textbox(label='Status', interactive=False)
                file_prev = gr.Textbox(label='Extracted Text Preview', lines=6, interactive=False)
                file_up.change(ui_load_file, file_up, [file_prev, file_st])
            with gr.Column(scale=1):
                d_model   = gr.Dropdown(choices=list(SUM_MODELS.keys()),
                                        value='BanglaBERT (Extractive) ⭐',
                                        label='Summarization Model')
                d_sum_btn = gr.Button('📝 Generate Summary', variant='primary')
                d_sum_out = gr.Textbox(label='Summary', lines=6, interactive=False)
                gr.Markdown('---')
                d_q       = gr.Textbox(label='Ask a Question (Bengali)',
                                       placeholder='প্রশ্ন লিখুন...')
                d_qa_mod  = gr.Dropdown(choices=QA_MODELS, value='BanglaBERT (FT) ⭐',
                                        label='QA Model')
                d_qa_btn  = gr.Button('🔍 Get Answer', variant='secondary')
                d_qa_out  = gr.Textbox(label='Answer', lines=3, interactive=False)
        d_sum_btn.click(ui_summarize_loaded, [d_model],        [d_sum_out])
        d_qa_btn.click( ui_qa_loaded,        [d_q, d_qa_mod],  [d_qa_out])

    # ── Tab 2: Image OCR ───────────────────────────────────────────────────
    with gr.Tab('🖼️ Image OCR'):
        gr.Markdown('Upload an image — Bengali text will be extracted via Tesseract OCR.')
        with gr.Row():
            with gr.Column(scale=1):
                img_up  = gr.Image(label='Upload Image', type='pil')
                img_st  = gr.Textbox(label='Status', interactive=False)
                img_txt = gr.Textbox(label='Extracted Text', lines=6, interactive=False)
                img_up.change(ui_load_image, [img_up], [img_txt, img_st])
            with gr.Column(scale=1):
                i_model   = gr.Dropdown(choices=list(SUM_MODELS.keys()),
                                        value='BanglaBERT (Extractive) ⭐',
                                        label='Summarization Model')
                i_sum_btn = gr.Button('📝 Summarize OCR Text', variant='primary')
                i_sum_out = gr.Textbox(label='Summary', lines=5, interactive=False)
                gr.Markdown('---')
                i_q       = gr.Textbox(label='Ask a Question',
                                       placeholder='প্রশ্ন লিখুন...')
                i_qa_mod  = gr.Dropdown(choices=QA_MODELS, value='BanglaBERT (FT) ⭐',
                                        label='QA Model')
                i_qa_btn  = gr.Button('🔍 Get Answer', variant='secondary')
                i_qa_out  = gr.Textbox(label='Answer', lines=3, interactive=False)
        i_sum_btn.click(ui_summarize_loaded, [i_model],        [i_sum_out])
        i_qa_btn.click( ui_qa_loaded,        [i_q, i_qa_mod],  [i_qa_out])

    # ── Tab 3: Direct Text Input ───────────────────────────────────────────
    with gr.Tab('✍️ Direct Text'):
        gr.Markdown('Paste or type Bengali text directly.')
        with gr.Row():
            with gr.Column(scale=1):
                t_ctx   = gr.Textbox(label='Bengali Text / Context', lines=10,
                                     placeholder='বাংলা লেখা এখানে পেস্ট করুন...')
            with gr.Column(scale=1):
                t_model   = gr.Dropdown(choices=list(SUM_MODELS.keys()),
                                        value='BanglaBERT (Extractive) ⭐',
                                        label='Summarization Model')
                t_sum_btn = gr.Button('📝 Summarize', variant='primary')
                t_sum_out = gr.Textbox(label='Summary', lines=5, interactive=False)
                gr.Markdown('---')
                t_q       = gr.Textbox(label='Question', placeholder='প্রশ্ন লিখুন...')
                t_qa_mod  = gr.Dropdown(choices=QA_MODELS, value='BanglaBERT (FT) ⭐',
                                        label='QA Model')
                t_qa_btn  = gr.Button('🔍 Get Answer', variant='secondary')
                t_qa_out  = gr.Textbox(label='Answer', lines=3, interactive=False)
        t_sum_btn.click(run_summarize, [t_ctx, t_model],       [t_sum_out])
        t_qa_btn.click( run_qa,        [t_ctx, t_q, t_qa_mod], [t_qa_out])

    # ── Tab 4: 🎙️ Speech to Text ──────────────────────────────────────────
    with gr.Tab('🎙️ Speech to Text'):
        gr.Markdown(
            '### 🎙️ Bengali Speech Recognition\n'
            'Record or upload audio — the Shrutimala ASR model transcribes it to Bengali. '
            'Then summarize or ask questions from the transcript.'
        )

        with gr.Row():
            # ── Left column: audio input & transcription ──────────────────
            with gr.Column(scale=1):
                sp_audio = gr.Audio(
                    label='🎤 Record / Upload Audio (WAV, MP3, OGG …)',
                    type='numpy',          # returns (sample_rate, np.ndarray)
                    sources=['microphone', 'upload'],
                )
                sp_btn   = gr.Button('🔁 Transcribe', variant='primary')
                sp_st    = gr.Textbox(label='Status', interactive=False, lines=1)
                sp_txt   = gr.Textbox(
                    label='📝 Transcribed Bengali Text',
                    lines=8,
                    placeholder='Transcription will appear here …',
                    interactive=True,     # user can edit if needed
                )
                sp_btn.click(ui_transcribe, [sp_audio], [sp_txt, sp_st])

            # ── Right column: summarize + QA from transcript ───────────────
            with gr.Column(scale=1):
                gr.Markdown('#### 📝 Summarize Transcript')
                sp_sum_model = gr.Dropdown(
                    choices=list(SUM_MODELS.keys()),
                    value='BanglaBERT (Extractive) ⭐',
                    label='Summarization Model'
                )
                sp_sum_btn = gr.Button('📝 Summarize', variant='primary')
                sp_sum_out = gr.Textbox(
                    label='Summary', lines=5, interactive=False
                )

                gr.Markdown('---\n#### 🔍 Ask a Question')
                sp_q       = gr.Textbox(
                    label='Question (Bengali)',
                    placeholder='প্রশ্ন লিখুন...'
                )
                sp_qa_mod  = gr.Dropdown(
                    choices=QA_MODELS,
                    value='BanglaBERT (FT) ⭐',
                    label='QA Model'
                )
                sp_qa_btn  = gr.Button('🔍 Get Answer', variant='secondary')
                sp_qa_out  = gr.Textbox(
                    label='Answer', lines=3, interactive=False
                )

        # wire buttons — pass sp_txt so user edits are respected
        sp_sum_btn.click(
            ui_speech_summarize,
            [sp_txt, sp_sum_model],
            [sp_sum_out]
        )
        sp_qa_btn.click(
            ui_speech_qa,
            [sp_txt, sp_q, sp_qa_mod],
            [sp_qa_out]
        )

    # ── Tab 5: Compare All Models ──────────────────────────────────────────
    with gr.Tab('🔬 Compare All Models'):
        gr.Markdown('Run the same input through every model at once.')
        with gr.Row():
            with gr.Column():
                cmp_ctx  = gr.Textbox(label='Context / Text', lines=6,
                                      placeholder='বাংলা লেখা এখানে...')
                cmp_q    = gr.Textbox(label='Question', placeholder='প্রশ্ন লিখুন...')
                cmp_btn  = gr.Button('⚡ Compare All QA Models',      variant='primary')
                cmp_sbtn = gr.Button('⚡ Compare All Summarizers',    variant='secondary')
        cmp_out = gr.Markdown()
        cmp_btn.click(
            run_qa,
            [cmp_ctx, cmp_q, gr.State('All Models (Comparison)')],
            [cmp_out]
        )
        cmp_sbtn.click(
            run_summarize,
            [cmp_ctx, gr.State('All Models (Comparison)')],
            [cmp_out]
        )
        gr.Examples(
            examples=[
                ['বাংলাদেশ ১৯৭১ সালের ২৬শে মার্চ স্বাধীনতা ঘোষণা করে। '
                 'শেখ মুজিবুর রহমান বাংলাদেশের প্রথম রাষ্ট্রপতি ছিলেন।',
                 'বাংলাদেশের প্রথম রাষ্ট্রপতি কে ছিলেন?'],
                ['রবীন্দ্রনাথ ঠাকুর ১৮৬১ সালের ৭ই মে কলকাতায় জন্মগ্রহণ করেন। '
                 'তিনি গীতাঞ্জলি কাব্যগ্রন্থের জন্য ১৯১৩ সালে নোবেল পুরস্কার পান।',
                 'রবীন্দ্রনাথ কোন বইয়ের জন্য নোবেল পুরস্কার পান?'],
            ],
            inputs=[cmp_ctx, cmp_q],
        )

    gr.HTML(
        '<div style="text-align:center;color:#888;padding:10px;font-size:0.8rem">'
        'Bengali NLP — BanglaBERT · XLM-RoBERTa · mT5-XLSum · '
        'BiLSTM · CNN · TextRank · TF-IDF · Shrutimala ASR'
        '</div>'
    )

demo.launch(share=True, debug=False)


Device: cuda
Loading models …


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  Loading BanglaBERT encoder (sagorsarker/bangla-bert-base) …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


All models loaded ✅
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10f03b10d65720135a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def extract_text_from_image(pil_img):
    return pytesseract.image_to_string(pil_img, lang='ben+eng', config='--psm 3')

In [ ]:
import cv2
import numpy as np
import pytesseract

def extract_text_from_image(pil_img):
    try:
        # Convert PIL → OpenCV
        img = np.array(pil_img)

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Improve clarity
        gray = cv2.bilateralFilter(gray, 9, 75, 75)

        # Threshold (very important for OCR)
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

        # OCR with better config
        text = pytesseract.image_to_string(
            thresh,
            lang='ben',
            config='--oem 3 --psm 6'
        )

        return text.strip()

    except Exception as e:
        return f"OCR Error: {e}"

In [ ]:
# Cell 9 ─ Full Gradio UI
import gradio as gr
import io
# ── Shared document state ──────────────────────────────────────────────────
doc_state = {'text': ''}

# ── Summarization dispatcher ───────────────────────────────────────────────
# All values are plain callables: fn(text) -> str
SUM_MODELS = {
    'mT5-XLSum (Abstractive) ⭐' : mt5_sum,
    'All Models (Comparison)'    : None,
    }

def run_summarize(text, model_choice):
    text = normalize_bengali(text.strip())
    if not text:
        return '⚠️ Please enter text.'
    if model_choice == 'All Models (Comparison)':
        parts = []
        for name, fn in list(SUM_MODELS.items())[:-1]:
            try:
                parts.append(f'**{name}:**\n{fn(text)}')
            except Exception as e:
                parts.append(f'**{name}:** ❌ {e}')
        return '\n\n---\n\n'.join(parts)
    fn = SUM_MODELS.get(model_choice)
    if fn is None:
        return '⚠️ Model not found.'
    try:
        return fn(text)
    except Exception as e:
        return f'❌ Error: {e}'

# ── QA dispatcher ─────────────────────────────────────────────────────────
QA_MODELS = [
    'BanglaBERT (FT) ⭐',
    'XLM-RoBERTa (Base)',
    'Zero-shot',
    'BiLSTM',
    'BiLSTM+Attention',
    'CNN+BiLSTM',
    'All Models (Comparison)',
]

def run_qa(context, question, model_choice):
    context  = context.strip()
    question = question.strip()
    if not context:  return '⚠️ Please provide context text.'
    if not question: return '⚠️ Please enter a question.'
    ctx_short = ' '.join(context.split()[:MAX_CTX])

    def _get(name):
        if 'BanglaBERT' in name:
            return bangla_pipe(question=question, context=context)['answer']
        elif 'XLM' in name:
            return xlmr_pipe(question=question, context=context)['answer']
        elif 'Zero' in name:
            return qa_zs(question=question, context=context)['answer']
        elif name == 'BiLSTM':
            return answer_pytorch(m1, ctx_short, question)
        elif name == 'BiLSTM+Attention':
            return answer_pytorch(m2, ctx_short, question)
        elif name == 'CNN+BiLSTM':
            return answer_pytorch(m3, ctx_short, question)
        return ''

    try:
        if model_choice == 'All Models (Comparison)':
            lines = []
            for nm in QA_MODELS[:-1]:
                ans = _get(nm)
                lines.append(f'**{nm}:** {ans or "(No answer found)"}')
            return '\n\n'.join(lines)
        return _get(model_choice) or '(No answer found)'
    except Exception as e:
        return f'❌ Error: {e}'

# ── File / Image loaders ───────────────────────────────────────────────────
def ui_load_file(file_obj):
    if file_obj is None: return '', '⚠️ No file uploaded.'
    try:
        with open(file_obj.name, 'rb') as f:
            raw = f.read()
        fname = os.path.basename(file_obj.name)
        text  = extract_text_from_file(raw, fname)
        if not text.strip(): return '', '⚠️ No text found.'
        doc_state['text'] = text
        preview = text[:1500] + ('...' if len(text) > 1500 else '')
        return preview, f'✅ {fname} ({len(text)} chars)'
    except Exception as e:
        return '', f'❌ {e}'

def ui_load_image(pil_img):
    if pil_img is None:
        return '', '⚠️ No image uploaded.'

    try:
        text = extract_text_from_image(pil_img)

        if not text.strip():
            return '', '⚠️ No text extracted.'

        doc_state['text'] = text

        preview = text[:1500] + ('...' if len(text) > 1500 else '')

        return preview, f'✅ OCR success ({len(text)} chars)'

    except Exception as e:
        return '', f'❌ OCR Error: {e}'

# ── UI wrappers for loaded-doc tabs ───────────────────────────────────────
def ui_summarize_loaded(sum_model):
    t = doc_state['text'].strip()
    if not t: return '⚠️ Please upload a document/image first.'
    return run_summarize(t, sum_model)

def ui_qa_loaded(question, qa_model):
    t = doc_state['text'].strip()
    if not t: return '⚠️ Please upload a document/image first.'
    return run_qa(t, question, qa_model)

# ── Build Gradio UI ────────────────────────────────────────────────────────
with gr.Blocks(title='Bengali NLP System') as demo:

    gr.HTML("""
    <div style='text-align:center;padding:16px 0'>
      <h1 style='color:#006a4e'>🇧🇩 Bengali NLP System</h1>
      <p style='color:#555'>Document QA &amp; Summarization &nbsp;|&nbsp; Image OCR &nbsp;|&nbsp; BanglaBERT · XLM-R · mT5 · BiLSTM</p>
    </div>
    """)

    # ── Tab 1: Document Upload ─────────────────────────────────────────────
    with gr.Tab('📄 Document'):
        gr.Markdown('Upload a **PDF / TXT / DOCX** file. Text is extracted automatically.')
        with gr.Row():
            with gr.Column(scale=1):
                file_up   = gr.File(label='Upload File (PDF / TXT / DOCX)')
                file_st   = gr.Textbox(label='Status', interactive=False)
                file_prev = gr.Textbox(label='Extracted Text Preview', lines=6, interactive=False)
                file_up.change(ui_load_file, file_up, [file_prev, file_st])
            with gr.Column(scale=1):
                d_model   = gr.Dropdown(choices=list(SUM_MODELS.keys()),
                                        value='mT5-XLSum (Abstractive) ⭐',
                                        label='Summarization Model')
                d_sum_btn = gr.Button('📝 Generate Summary', variant='primary')
                d_sum_out = gr.Textbox(label='Summary', lines=6, interactive=False)
                gr.Markdown('---')
                d_q       = gr.Textbox(label='Ask a Question (Bengali)',
                                       placeholder='প্রশ্ন লিখুন...')
                d_qa_mod  = gr.Dropdown(choices=QA_MODELS, value='BanglaBERT (FT) ⭐',
                                        label='QA Model')
                d_qa_btn  = gr.Button('🔍 Get Answer', variant='secondary')
                d_qa_out  = gr.Textbox(label='Answer', lines=3, interactive=False)
        d_sum_btn.click(ui_summarize_loaded, [d_model],     [d_sum_out])
        d_qa_btn.click( ui_qa_loaded,        [d_q, d_qa_mod], [d_qa_out])

    # ── Tab 2: Image OCR ───────────────────────────────────────────────────
    with gr.Tab('🖼️ Image OCR'):
        gr.Markdown('Upload an image — Bengali text will be extracted via Tesseract OCR.')
        with gr.Row():
            with gr.Column(scale=1):
                img_up  = gr.Image(label='Upload Image', type='pil')
                img_st  = gr.Textbox(label='Status', interactive=False)
                img_txt = gr.Textbox(label='Extracted Text', lines=6, interactive=False)
                img_up.change(ui_load_image, [img_up], [img_txt, img_st])
            with gr.Column(scale=1):
                i_model   = gr.Dropdown(choices=list(SUM_MODELS.keys()),
                                        value='mT5-XLSum (Abstractive) ⭐',
                                        label='Summarization Model')
                i_sum_btn = gr.Button('📝 Summarize OCR Text', variant='primary')
                i_sum_out = gr.Textbox(label='Summary', lines=5, interactive=False)
                gr.Markdown('---')
                i_q       = gr.Textbox(label='Ask a Question',
                                       placeholder='প্রশ্ন লিখুন...')
                i_qa_mod  = gr.Dropdown(choices=QA_MODELS, value='BanglaBERT (FT) ⭐',
                                        label='QA Model')
                i_qa_btn  = gr.Button('🔍 Get Answer', variant='secondary')
                i_qa_out  = gr.Textbox(label='Answer', lines=3, interactive=False)
        i_sum_btn.click(ui_summarize_loaded, [i_model],       [i_sum_out])
        i_qa_btn.click( ui_qa_loaded,        [i_q, i_qa_mod], [i_qa_out])

    # ── Tab 3: Direct Text Input ───────────────────────────────────────────
    with gr.Tab('✍️ Direct Text'):
        gr.Markdown('Paste or type Bengali text directly.')
        with gr.Row():
            with gr.Column(scale=1):
                t_ctx   = gr.Textbox(label='Bengali Text / Context', lines=10,
                                     placeholder='বাংলা লেখা এখানে পেস্ট করুন...')
            with gr.Column(scale=1):
                t_model   = gr.Dropdown(choices=list(SUM_MODELS.keys()),
                                        value='mT5-XLSum (Abstractive) ⭐',
                                        label='Summarization Model')
                t_sum_btn = gr.Button('📝 Summarize', variant='primary')
                t_sum_out = gr.Textbox(label='Summary', lines=5, interactive=False)
                gr.Markdown('---')
                t_q       = gr.Textbox(label='Question', placeholder='প্রশ্ন লিখুন...')
                t_qa_mod  = gr.Dropdown(choices=QA_MODELS, value='BanglaBERT (FT) ⭐',
                                        label='QA Model')
                t_qa_btn  = gr.Button('🔍 Get Answer', variant='secondary')
                t_qa_out  = gr.Textbox(label='Answer', lines=3, interactive=False)
        t_sum_btn.click(run_summarize, [t_ctx, t_model],          [t_sum_out])
        t_qa_btn.click( run_qa,        [t_ctx, t_q, t_qa_mod],    [t_qa_out])

    # ── Tab 4: Model Comparison ────────────────────────────────────────────
    with gr.Tab('🔬 Compare All Models'):
        gr.Markdown('Run the same input through every model at once.')
        with gr.Row():
            with gr.Column():
                cmp_ctx = gr.Textbox(label='Context / Text', lines=6,
                                     placeholder='বাংলা লেখা এখানে...')
                cmp_q   = gr.Textbox(label='Question', placeholder='প্রশ্ন লিখুন...')
                cmp_btn = gr.Button('⚡ Compare All QA Models', variant='primary')
                cmp_sbtn= gr.Button('⚡ Compare All Summarizers', variant='secondary')
        cmp_out = gr.Markdown()
        cmp_btn.click( run_qa,        [cmp_ctx, cmp_q,
                        gr.State('All Models (Comparison)')], [cmp_out])
        cmp_sbtn.click(run_summarize, [cmp_ctx,
                        gr.State('All Models (Comparison)')], [cmp_out])
        gr.Examples(
            examples=[
                ['বাংলাদেশ ১৯৭১ সালের ২৬শে মার্চ স্বাধীনতা ঘোষণা করে। '
                 'শেখ মুজিবুর রহমান বাংলাদেশের প্রথম রাষ্ট্রপতি ছিলেন।',
                 'বাংলাদেশের প্রথম রাষ্ট্রপতি কে ছিলেন?'],
                ['রবীন্দ্রনাথ ঠাকুর ১৮৬১ সালের ৭ই মে কলকাতায় জন্মগ্রহণ করেন। '
                 'তিনি গীতাঞ্জলি কাব্যগ্রন্থের জন্য ১৯১৩ সালে নোবেল পুরস্কার পান।',
                 'রবীন্দ্রনাথ কোন বইয়ের জন্য নোবেল পুরস্কার পান?'],
            ],
            inputs=[cmp_ctx, cmp_q],
        )

    gr.HTML('<div style="text-align:center;color:#888;padding:10px;font-size:0.8rem">'
            'Bengali NLP — BanglaBERT · XLM-RoBERTa · mT5-XLSum · BiLSTM · CNN+BiLSTM · TF-IDF'
            '</div>')

demo.launch(share=True, debug=False)


In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-ben

In [ ]:
import os
os.environ['TESSDATA_PREFIX'] = '/usr/share/tesseract-ocr/4.00/tessdata/'

In [ ]:
from google.colab import files
from PIL import Image
import pytesseract

uploaded = files.upload()

In [ ]:

img_path = list(uploaded.keys())[0]
img = Image.open(img_path)

In [ ]:
text = pytesseract.image_to_string(img, lang='ben')
print("📝 Extracted Text:\n")
print(text)